# Practical 08 — Packing the Service into a Container
*A Dockerfile, an image, and a service that runs the same everywhere*

**SCSE3040 Machine Learning Operations · Bennett University · Session 2026-27**

| | |
|---|---|
| **Name** | `<YOUR NAME>` |
| **Roll Number** | `<YOUR ROLL NUMBER>` |
| **Batch** | `<YOUR BATCH>` |

## Aim
- Explain what a container gives you that a virtual environment cannot.
- Write a Dockerfile for the delivery-time service.
- Build an image and run it, then call the service inside it.
- Read a container's logs and shut it down cleanly.

## What a container gives you that a virtual environment cannot
A virtual environment (P01) pins only the **Python libraries**. It does not pin the Python
interpreter version, the operating system, the system libraries underneath, or anything else
that happens to be installed on the machine. A **container** packs all of that (code, libraries,
Python and a minimal OS userland) into one sealed image. Only the host kernel is shared, so the
image runs the same on any machine with Docker.

- **Image**: the packed box, built once and stored.
- **Container**: a running copy of an image (one image, many containers, like class and objects).
- **Dockerfile**: the recipe that builds the image, read from top to bottom.
- **Layers**: every instruction makes a cached layer. Copying `requirements.txt` and installing
  libraries *before* copying code means a code change reuses the cached library layer.

In [1]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'fastapi', 'joblib', 'requests']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'fastapi', 'joblib', 'requests']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.13.7
Folder  : labs


  numpy          ok
  pandas         ok
  sklearn        ok
  fastapi        ok
  joblib         ok
  requests       ok

All good. You can carry on to Step 1.


### Step 0b — the dataset
Every practical in this course uses the same 600 food deliveries. The next cell makes sure the file is there.

In [2]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> ..\data\delivery_times.csv


## Walkthrough
### Step 1 — Check Docker is running before anything else

In [3]:
import shutil
import subprocess
from pathlib import Path

WORK = Path("work")
WORK.mkdir(exist_ok=True)

IMAGE = "delivery-api:1.0"
CONTAINER = "delivery-lab"
PORT = 8010

DOCKER_OK = False
if shutil.which("docker") is None:
    print("Docker is NOT installed on this machine.")
    print("Install Docker Desktop, then restart Jupyter.")
else:
    probe = subprocess.run(["docker", "version", "--format",
                            "{{.Server.Version}}"],
                           capture_output=True, text=True)
    if probe.returncode == 0 and probe.stdout.strip():
        DOCKER_OK = True
        print("Docker server version:", probe.stdout.strip())
    else:
        print("Docker is installed but not running.")
        print("Start Docker Desktop and wait for the whale icon")
        print("to stop animating, then run this cell again.")

print()
print("DOCKER_OK =", DOCKER_OK)

Docker server version: 29.9.2

DOCKER_OK = True


Every later cell checks `DOCKER_OK` before running a Docker command.

### Step 2 — A helper for running Docker commands

In [4]:
def docker(*args, timeout=900, quiet=False):
    """Run one docker command and show its output."""
    if not DOCKER_OK:
        print(f"SKIPPED (no Docker): docker {' '.join(args)}")
        return None
    print(f"$ docker {' '.join(args)}")
    try:
        done = subprocess.run(["docker", *args], timeout=timeout,
                              capture_output=True, text=True,
                              encoding="utf-8", errors="replace")  # Windows-safe
    except subprocess.TimeoutExpired:
        print(f"timed out after {timeout} seconds")
        return None
    output = (done.stdout + done.stderr).strip()
    if output and not quiet:
        print(output[-2500:])
    print("exit code:", done.returncode)
    print("-" * 60)
    return done


docker("--version")

$ docker --version
Docker version 29.9.2, build a82e528
exit code: 0
------------------------------------------------------------


CompletedProcess(args=['docker', '--version'], returncode=0, stdout='Docker version 29.9.2, build a82e528\n', stderr='')

### Step 3 — Gather what goes in the box
The service needs three things: the trained model, the application code, and a list of libraries.

In [5]:
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
orders = pd.read_csv(DATA)
X_train, _, y_train, _ = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
joblib.dump(LinearRegression().fit(X_train, y_train),
            WORK / "model.joblib")

app_py = '''"The delivery-time prediction service."

from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

HERE = Path(__file__).resolve().parent
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]

app = FastAPI(title="Delivery Time Predictor", version="1.0.0")
model = joblib.load(HERE / "model.joblib")


class Order(BaseModel):
    distance_km: float = Field(..., gt=0, le=50)
    prep_time_min: int = Field(..., ge=0, le=120)
    traffic_level: int = Field(..., ge=1, le=3)
    rain: int = Field(..., ge=0, le=1)


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}


@app.post("/predict")
def predict(order: Order):
    row = pd.DataFrame([order.model_dump()])[FEATURES]
    return {"delivery_min": round(float(model.predict(row)[0]), 1),
            "model_version": app.version}
'''
(WORK / "app.py").write_text(app_py, encoding="utf-8")

requirements = "\n".join([
    "fastapi==0.141.1",
    "uvicorn==0.52.1",
    "scikit-learn==1.9.0",
    "pandas==2.3.3",
    "joblib==1.5.3",
]) + "\n"
(WORK / "requirements.txt").write_text(requirements, encoding="utf-8")

for f in sorted(WORK.iterdir()):
    if f.is_file():
        print(f"  {f.name:<20} {f.stat().st_size:>9,} bytes")

  .dockerignore               72 bytes
  app.py                     944 bytes
  Dockerfile                 386 bytes
  model.joblib               967 bytes
  requirements.txt            86 bytes


### Step 4 — Write the Dockerfile
- `FROM` picks the starting point; `slim` is a stripped-down Debian with Python.
- `WORKDIR` is the folder inside the box where everything happens.
- `COPY requirements.txt` **before** the code: the layer-cache trick.
- `--no-cache-dir` stops pip keeping downloads we will never reuse.
- `EXPOSE` documents the port; `-p` actually publishes it.
- `CMD ... --host 0.0.0.0` is essential, or nothing outside the box can reach the server.

In [6]:
dockerfile = """# The delivery-time prediction service.
FROM python:3.12-slim

WORKDIR /app

# Libraries first: this layer is cached and rarely changes.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Code and model last: these change often.
COPY app.py .
COPY model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(WORK / "Dockerfile").write_text(dockerfile, encoding="utf-8")
print((WORK / "Dockerfile").read_text(encoding="utf-8"))

# The delivery-time prediction service.
FROM python:3.12-slim

WORKDIR /app

# Libraries first: this layer is cached and rarely changes.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Code and model last: these change often.
COPY app.py .
COPY model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



### Step 5 — Tell Docker what to ignore
A `.dockerignore` keeps caches, notebooks and virtual environments out of the build context.

In [7]:
dockerignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
*.ipynb
.venv/
.git/
*.db
"""

(WORK / ".dockerignore").write_text(dockerignore, encoding="utf-8")
print((WORK / ".dockerignore").read_text(encoding="utf-8"))

__pycache__/
*.pyc
.ipynb_checkpoints/
*.ipynb
.venv/
.git/
*.db



### Step 6 — Build the image
`-t` names and tags the image. The first build takes several minutes; later builds reuse cached layers.

In [8]:
build = docker("build", "-t", IMAGE, str(WORK))

$ docker build -t delivery-api:1.0 work


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 425B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 2.0s

#3 [internal] load .dockerignore
#3 transferring context: 112B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 2.12kB done
#4 DONE 0.0s

#5 [1/6] FROM docker.io/library/python:3.12-slim@sha256:12316debbfb86890f6b2d825820612741ba527ad5a44e3902759141d572bfe8f
#5 resolve docker.io/library/python:3.12-slim@sha256:12316debbfb86890f6b2d825820612741ba527ad5a44e3902759141d572bfe8f 0.0s done
#5 DONE 0.0s

#6 [2/6] WORKDIR /app
#6 CACHED

#7 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#7 CACHED

#8 [3/6] COPY requirements.txt .
#8 CACHED

#9 [5/6] COPY app.py .
#9 CACHED

#10 [6/6] COPY model.joblib .
#10 CACHED

#11 exporting to image
#11 exporting layers done
#11 exporting manifest sha256:a9394566409c7e5fd6d6

Watch the output for the word **CACHED**: each one is a layer Docker did not have to redo.

### Step 7 — Look at what you built

In [9]:
docker("images", IMAGE)

$ docker images delivery-api:1.0
IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA
delivery-api:1.0   bb5505f6ec82        683MB          156MB
exit code: 0
------------------------------------------------------------


CompletedProcess(args=['docker', 'images', 'delivery-api:1.0'], returncode=0, stdout='IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA\ndelivery-api:1.0   bb5505f6ec82        683MB          156MB        \n', stderr='')

Most of the size is scikit-learn, pandas and numpy, not our code. Starting from
`python:3.12` instead of `python:3.12-slim` would add roughly 300 MB for no gain.

### Step 8 — Run it
`-d` runs it in the background, `--rm` deletes the container when it stops, and
`-p 8010:8000` connects port **8010 on the laptop** to port **8000 inside the box**.

In [10]:
import time

docker("rm", "-f", CONTAINER, quiet=True)      # clear any leftover
run = docker("run", "-d", "--rm", "--name", CONTAINER,
             "-p", f"{PORT}:8000", IMAGE)

if DOCKER_OK:
    time.sleep(6)          # give uvicorn a moment to start
    docker("ps", "--filter", f"name={CONTAINER}")

$ docker rm -f delivery-lab
exit code: 0
------------------------------------------------------------
$ docker run -d --rm --name delivery-lab -p 8010:8000 delivery-api:1.0


af84cc4e4d976e03e9ce9eea6d0c428324da3a093057ddc3e9975cb976fd4263
exit code: 0
------------------------------------------------------------


$ docker ps --filter name=delivery-lab
CONTAINER ID   IMAGE              COMMAND                  CREATED         STATUS         PORTS                                         NAMES
af84cc4e4d97   delivery-api:1.0   "uvicorn app:app --h…"   7 seconds ago   Up 6 seconds   0.0.0.0:8010->8000/tcp, [::]:8010->8000/tcp   delivery-lab
exit code: 0
------------------------------------------------------------


### Step 9 — Talk to the service inside the container

In [11]:
import requests

if DOCKER_OK:
    base = f"http://127.0.0.1:{PORT}"
    try:
        health = requests.get(f"{base}/health", timeout=10)
        print("GET /health ->", health.status_code, health.json())

        order = {"distance_km": 5.0, "prep_time_min": 20,
                 "traffic_level": 2, "rain": 0}
        pred = requests.post(f"{base}/predict", json=order,
                             timeout=10)
        print("POST /predict ->", pred.status_code, pred.json())
    except requests.RequestException as problem:
        print("Could not reach the container:", problem)
        print("Run the previous cell again, wait, then retry.")
else:
    print("SKIPPED (no Docker): nothing is listening on port",
          PORT)

GET /health -> 200 {'status': 'ok', 'model_loaded': True}
POST /predict -> 200 {'delivery_min': 43.0, 'model_version': '1.0.0'}


That prediction came from a model inside a container. Copy the image to any machine with Docker and you get the same answer.

### Step 10 — Read its logs, then stop it

In [12]:
docker("logs", "--tail", "15", CONTAINER)
docker("stop", CONTAINER)

print()
print("Containers still running:")
docker("ps")

$ docker logs --tail 15 delivery-lab


INFO:     172.17.0.1:57750 - "GET /health HTTP/1.1" 200 OK
INFO:     172.17.0.1:57752 - "POST /predict HTTP/1.1" 200 OK
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
exit code: 0
------------------------------------------------------------
$ docker stop delivery-lab


delivery-lab
exit code: 0
------------------------------------------------------------

Containers still running:
$ docker ps
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
exit code: 0
------------------------------------------------------------


CompletedProcess(args=['docker', 'ps'], returncode=0, stdout='CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES\n', stderr='')

The service logged each request. `docker logs` reads it, so there is no log file to open.

---
## Your turn
### Task T1 — Keep the rubbish out of your image

In [13]:
T1_patterns = ["__pycache__/", "*.pyc", ".venv/", ".git/", "tests/", "*.md"]

(WORK / ".dockerignore.mine").write_text("\n".join(T1_patterns) + "\n",
                                         encoding="utf-8")

print(T1_patterns)
print((WORK / ".dockerignore.mine").read_text(encoding="utf-8"))

['__pycache__/', '*.pyc', '.venv/', '.git/', 'tests/', '*.md']
__pycache__/
*.pyc
.venv/
.git/
tests/
*.md



### Task T2 — Fix a broken Dockerfile
**Diagnosis of the three faults:**
1. `COPY . .` runs before `pip install`, so any code change invalidates the library layer and
   every rebuild reinstalls everything. **Fix:** copy `requirements.txt` alone, install, *then* copy `app.py` and the model.
2. `uvicorn` listens on `127.0.0.1` by default, which is only inside the container. **Fix:** add `--host 0.0.0.0`.
3. No `WORKDIR`, and `COPY . .` dumps the whole folder into the image root `/`. **Fix:** `WORKDIR /app` and copy only the files the service needs.

In [14]:
T2_broken = '''FROM python:3.12-slim
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
EXPOSE 8000
CMD ["uvicorn", "app:app", "--port", "8000"]
'''

print("THE BROKEN ONE:")
print(T2_broken)

T2_fixed = """FROM python:3.12-slim

# Fix 3: a working folder, so files do not land in the image root.
WORKDIR /app

# Fix 1: install libraries before copying code, so this layer stays cached.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model.joblib .

EXPOSE 8000

# Fix 2: listen on every interface, not just inside the container.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print("YOURS:")
print(T2_fixed)

THE BROKEN ONE:
FROM python:3.12-slim
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
EXPOSE 8000
CMD ["uvicorn", "app:app", "--port", "8000"]

YOURS:
FROM python:3.12-slim

# Fix 3: a working folder, so files do not land in the image root.
WORKDIR /app

# Fix 1: install libraries before copying code, so this layer stays cached.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model.joblib .

EXPOSE 8000

# Fix 2: listen on every interface, not just inside the container.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



### Task T3 — Rebuild and serve a rainy order

In [15]:
T3_minutes = None

if not DOCKER_OK:
    print("SKIPPED: this task needs Docker Desktop running.")
else:
    (WORK / "Dockerfile.v2").write_text(T2_fixed, encoding="utf-8")
    docker("build", "-f", str(WORK / "Dockerfile.v2"),
           "-t", "delivery-api:2.0", str(WORK))

    docker("rm", "-f", "delivery-lab-2", quiet=True)
    docker("run", "-d", "--rm", "--name", "delivery-lab-2",
           "-p", "8011:8000", "delivery-api:2.0")
    time.sleep(6)

    rainy = {"distance_km": 4, "prep_time_min": 30,
             "traffic_level": 3, "rain": 1}
    reply = requests.post("http://127.0.0.1:8011/predict", json=rainy,
                          timeout=10)
    print("POST /predict ->", reply.status_code, reply.json())
    T3_minutes = reply.json()["delivery_min"]

    docker("stop", "delivery-lab-2")

print("T3_minutes =", T3_minutes)

$ docker build -f work\Dockerfile.v2 -t delivery-api:2.0 work


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.v2
#1 transferring dockerfile: 495B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 0.8s

#3 [internal] load .dockerignore
#3 transferring context: 112B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 95B done
#4 DONE 0.0s

#5 [1/6] FROM docker.io/library/python:3.12-slim@sha256:12316debbfb86890f6b2d825820612741ba527ad5a44e3902759141d572bfe8f
#5 resolve docker.io/library/python:3.12-slim@sha256:12316debbfb86890f6b2d825820612741ba527ad5a44e3902759141d572bfe8f 0.0s done
#5 DONE 0.0s

#6 [2/6] WORKDIR /app
#6 CACHED

#7 [3/6] COPY requirements.txt .
#7 CACHED

#8 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#8 CACHED

#9 [5/6] COPY app.py .
#9 CACHED

#10 [6/6] COPY model.joblib .
#10 CACHED

#11 exporting to image
#11 exporting layers done
#11 exporting manifest sha256:a9394566409c7e5

0dcbc3efd951d13c9514aad62c72cd655346e239ef6754e62129b2a9cbbf0bac
exit code: 0
------------------------------------------------------------


POST /predict -> 200 {'delivery_min': 56.1, 'model_version': '1.0.0'}
$ docker stop delivery-lab-2


delivery-lab-2
exit code: 0
------------------------------------------------------------
T3_minutes = 56.1


## Self-check

In [16]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | the file exists', lambda: (WORK / '.dockerignore.mine').is_file())
_check('T1 | it contains all six patterns, one per line', lambda: set(ln.strip() for ln in (WORK / '.dockerignore.mine').read_text(encoding='utf-8').splitlines() if ln.strip()) == {'__pycache__/', '*.pyc', '.venv/', '.git/', 'tests/', '*.md'})
_check('T1 | T1_patterns lists the same six', lambda: set(T1_patterns) == {'__pycache__/', '*.pyc', '.venv/', '.git/', 'tests/', '*.md'})
_check('T2 | the fixed Dockerfile sets a WORKDIR', lambda: 'WORKDIR' in T2_fixed)
_check('T2 | requirements are installed before the code is copied', lambda: T2_fixed.index('pip install') < T2_fixed.index('COPY app.py'))
_check('T2 | uvicorn is told to listen on 0.0.0.0', lambda: '--host' in T2_fixed and '0.0.0.0' in T2_fixed)
_check('T2 | it still starts from a slim python base and exposes a port', lambda: 'FROM python:3.12-slim' in T2_fixed and 'EXPOSE' in T2_fixed)
_check('T3 | the second Dockerfile was written', lambda: (WORK / 'Dockerfile.v2').is_file())
_check('T3 | the container returned a sensible prediction', lambda: 30 < float(T3_minutes) < 70)

print("==================================================================")
print("SELF-CHECK   Practical 08 --- Packing the Service into a Container")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

SELF-CHECK   Practical 08 --- Packing the Service into a Container
  [PASS]  T1 | the file exists
  [PASS]  T1 | it contains all six patterns, one per line
  [PASS]  T1 | T1_patterns lists the same six
  [PASS]  T2 | the fixed Dockerfile sets a WORKDIR
  [PASS]  T2 | requirements are installed before the code is copied
  [PASS]  T2 | uvicorn is told to listen on 0.0.0.0
  [PASS]  T2 | it still starts from a slim python base and exposes a port
  [PASS]  T3 | the second Dockerfile was written
  [PASS]  T3 | the container returned a sensible prediction
------------------------------------------------------------------
  9 of 9 checks passed
Well done. Save the notebook and submit it.
